In [ ]:
""" 
    1. class Common_Dataset
        공용 DS 모델
    
    2. class MyModel
        동적 은닉층 모델(linear only)
    
    3. class TT_classifier
        training()  train
        evaluate()  test용 함수.
    
    4. class TT_regressior
        training()  train
        evaluate()  test용 함수.
"""

' \n    1. class Module_DNN\n        학습 모델 제작\n    \n    2. class Module_TT\n        training()  train\n        evaluate()  test용 함수.\n'

In [1]:
import torch                                            ## Tensor 및 기본 함수들 관련 모듈들
import torch.nn as nn                                   ## 인공신경망 관련 모듈들
import torch.nn.functional as F                         ## 인공신경망 관련 함수들
import torch.optim as optim                             ## 최적화 모듈
from torch.utils.data import Dataset, DataLoader 

import torchmetrics
import torchinfo

from sklearn.model_selection import train_test_split    ## 학습용 데이터셋 관련 함수

In [4]:
## 범용 데이터셋 클래스 
class Common_Dataset(Dataset):
    # 피쳐와 타겟 분리 및 전처리 진행 
    def __init__(self, featureDF, targetSR):
        super().__init__()
        self.feature = featureDF
        self.target  = targetSR
        self.rows = featureDF.shape[0]
        self.cols = featureDF.shape[1]
    
    # 데이터셋의 샘플 수 반환 메서드 
    def __len__(self):
        return self.rows 

    # DataLoader에서 batch_size만큼 호출하는 메서드
    # 인덱스에 해당하는 피쳐와 타겟 반환 단, Tensor 형태
    def __getitem__(self, index):
       arrFeature = self.feature.iloc[index].values   # ndarray
       arrTarget = self.target[index].reshape(-1)     # ndarray
   
       return torch.FloatTensor(arrFeature), torch.Tensor(arrTarget)

In [3]:
# 은닉층 개수 동적인 모델 ---------------------------------------------------------------------
class MyModel(nn.Module):
    def __init__(self, in_in, out_out, h_in, h_list=[]):
        """ 
        in_in : 입력층의 입력. 피처 개수
        h_in : 입력층 출력. 최초 히든입력.
        out_out : 최종 출력값.
        h_list = [] 리스트
        """
        # 부모클래스 생성
        super().__init__()
        # 자식클래스의 인스턴스 속성 설정
        self.input_layer = nn.Linear(in_in, h_in)
        
        self.h1_layer = nn.ModuleDict() 
        for idx in range(len(h_list)):
            h_in = h_list[idx-1] if idx else h_in
            h_out = h_list[idx]
            self.h1_layer[f"hl_{str(idx)}"] = nn.Linear(h_in,h_out)
            
        self.output_layer = nn.Linear(h_out, out_out)
        
    def forward(self, x):
        y=F.relu(self.input_layer(x))
    
        for linear in self.h1_layer.values():
            y=F.relu(linear(y))
            
        return self.output_layer(y)

In [ ]:
class TT_classifier():
    """ 
    데이터를 텐서 타입으로 바꾸어서 넣어야함.
    그 중에서도 데이터타입
    
    ## [5-1] 학습 관련 설정들
    EPOCH = 1000                                         # 학습용 DS를 처음부터 끝까지 1번 학습하는 것을 에포크
    BATCH_SIZE = 1200                                      # DS를 학습량 만큼 나눈 사이즈
    ITERATION = int(X_train.shape[0]/BATCH_SIZE)         # 학습용 DS이 분리된 수 => 1에포크에 W, b 업데이트 횟수
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    # [5-2] 학습 관련 인스턴스들
    LR      = 0.0001                                # Learning Rate
    MODEL  = Model()                           #학습 모델
    OPTIMIZER = optim.Adam(MODEL.parameters(), lr=LR)      #최적화 즉, 경사하강법 알고리즘으로 W, b의 값 개신
    LOSS_FN = nn.CrossEntropyLoss()
 """
    def __init__(self, model, trainDS, testDS, lr=0.001, batch_size=100, epoch=100, device=None):
            self.TRAINDL   = DataLoader(trainDS, batch_size=batch_size) ## 학습용 데이터로더
            self.TESTDL    = DataLoader(testDS,  batch_size=batch_size) ## 테스트용 데이터로더
            self.EPOCH = epoch
            self.BATCH_SIZE = batch_size
            self.ITERATION = int(trainDS.shape[0]/batch_size)
            self.DEVICE = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')

            self.LR = lr
            self.MODEL = model.to(self.DEVICE)
            self.OPTIMIZER = optim.Adam(self.MODEL.parameters(), lr=self.LR)
            self.LOSS_FN = nn.CrossEntropyLoss()
            
            
    def training(self):
    # 학습 모드 설정
        self.MODEL.train()

        E_LOSS, E_ACC = 0, 0
        for feature, target in self.TRAINDL:
            # 배치크기만큼 feature, target로딩
            #print('로딩 데이터 :', feature.shape, target.shape)

            # 가중치 기울기 0 초기화
            self.OPTIMIZER.zero_grad()

            # 학습 진행
            pre_y = self.MODEL(feature)

            # 손실 계산
            loss = self.LOSS_FN(pre_y, target.reshape(-1).long())

            # 정확도 계산
            accuarcy = MulticlassAccuracy(num_classes=3)
            acc = accuarcy(pre_y, target.reshape(-1))

            # 역전파 진행
            loss.backward()

            # 가중치/절편 업데이트
            self.OPTIMIZER.step()

            E_LOSS += loss.item()
            E_ACC  += acc.item()

        return E_LOSS/self.ITERATION, E_ACC/self.ITERATION
      
    def evaluate(self):
        # 에포크 단위로 검증 => 검증 모드
        self.MODEL.eval()
        
        # W, b가 업데이트 해제
        with torch.no_grad():
            # 검증용 데이터셋 => 텐서화 ndarray ==> tensor변환
            x = torch.FloatTensor(X_test.values) 
            y = torch.Tensor(y_test.values)
            
            # 검증진행
            pre_y= self.MODEL(x)
            
            # 손실 계산
            loss = self.LOSS_FN(pre_y, y.reshape(-1).long())

            # 정확도 계산
            accuarcy = MulticlassAccuracy(num_classes=3)
            acc = accuarcy(pre_y, y.reshape(-1))

            return loss.item(), acc.item()